# Submit a Parameter Ensemble

To recap what we've done so far, we created a set of parameter files with `meristem.paramgen`: 6 FATES parameter files 
(`fates_oat_0000.json`...`fates_oat_0005.json`). Each one sets one of 3 parameters to its minimum or maximum value, as defined in `<meristem>/examples/paramgen/fates_example/fates_param_dir/main.csv`.

Next, we will use `meristem.ensemble` to run a CLM-FATES case with each of those parameter files, and to track and manage the whole ensemble. 

## meristem.ensemble

`meristem.ensemble` handles creation, submission, and tracking/management of parameter ensembles of CLM and CLM-FATES. Each ensemble member is a clone of one already-built case. `meristem` sets each member's parameter file in the clone and submits it, so nothing needs to be rebuilt. 

<div class="alert alert-warning">
  <strong>Important Note:</strong> All of the command-line steps in this notebook should be done on Derecho, not from Casper. The Python cells that write the script and YAML can still run in Jupyter on Casper, since they share the same filesystem.
</div>

### Requirements

To run `meristem.ensemble` you need:

- A **built base case**: for us, `BONA.FATESsp` (see Step 1)
- **Paramgen output**: for us, `/glade/u/home/$USER/paramfiles_OAT`
- `meristem` environment: for us, the `meristem` conda env from the [overview notebook](sensitivity_overview.ipynb)

## Step 1: Check (or generate) the base case

As our base case, we will use the `BONA.FATESsp` case from the [single-point exercises](../singlepoint/exercises/FATESsp_NEON.ipynb). Navigate to that directory:

```bash
cd /glade/u/home/$USER/cases/BONA.FATESsp
```

In order for `meristem.ensemble` to clone this case (`./create_clone` from `cime/scripts`), the case must be built. Check that the build is complete:

```bash
./xmlquery BUILD_COMPLETE
```

This should give `TRUE`. If not, run `qcmd -- ./case.build` inside the case directory. Or, if you skipped this exercise, complete it now.

We will also use this base case as our "default" run for comparison against our ensemble of parameter-modified runs. So if you haven't run this case yet, be sure to do that as well (`./case.submit`).

Together with our 6 members it makes up the 2N+1 runs of a one-at-a-time experiment. The members start in 2014 like the base case, so we'll compare them over 2014–2015.

---

Because we are cloning this case for each of our ensemble members, each of our clones will inherit the FATES-SP settings, the `hist_fincl2` we added to the `user_nl_clm`, and all of the `xmlchange` operations we completed in the exercise.


<div class="alert alert-warning">
  <strong>Important Note:</strong> meristem doesn't create the member cases immediately upon <code>ensemble create</code> or <code>ensemble submit</code>. This happens in several member-specific setup jobs. So if you modify any xml or namelist variables in the base case before the setup jobs finish, they will carry over to your member clones.
  
  Be sure to make any edits to the base case <em>after</em> the setup jobs have finished. Or, for safety, don't make any edits to the base case once you have initialized your ensemble.
</div>

## Step 2: Write a stage script

Sometimes, we want to make modifications to our clones from the base case. `meristem.ensemble` facilitates this through running an optional user-supplied script during the case creation phase of each member case. After each clone is created and set up (`./case.setup`), `meristem` runs the user-supplied script. Then `meristem` sets the member's parameter file and submits the case.

This script is thus the place for any `xmlchange` or `user_nl_clm` modifications, but **should not** set the `paramfile` or `fates_paramfile`, nor should it run `./case.submit`. `meristem` handles both of these operations.

The script is called from inside each members `root` directory, and is called with `bash <script> <case_root> <previous_case_name> <previous_case_root>`.

The `<previous_case_name>` and `<previous_case_root>` are used in "tethering" different model stages together ([see the meristem run documentation](https://github.com/adrifoster/meristem/blob/main/docs/run.md)), but we will not make use of that feature here.

Because the script is called from each member's `root` directory (not its case directory) the script should first `cd` into the case directory with `cd $1`.

The below code creates a small script that `meristem` will run once for each member case. Run it here to create one we can use.

Be sure to change the `user` variable to your username on the NCAR machines.

In [ ]:
from pathlib import Path

user = "me" # add your username here

script = """\
#!/usr/bin/env bash

if [ $# -lt 1 ]; then
  echo "ERROR: please specify a case_root"
  exit 1
fi
case_root="$1"

cd ${case_root}

./xmlchange --subgroup case.run JOB_WALLCLOCK_TIME=01:00:00
./xmlchange --subgroup case.st_archive JOB_WALLCLOCK_TIME=01:00:00
./xmlchange STOP_OPTION="nyears"
./xmlchange STOP_N=2
./xmlchange RESUBMIT=0
./xmlchange CLM_FORCE_COLDSTART=on
"""

script_path = Path(f"/glade/u/home/{user}/code/setup_oat.sh")
script_path.write_text(script)
print(f"Wrote {script_path}")

Notice that we:

1. Used the first argument to the script (`$1`, the case directory) to change into the case directory *before* making any xml modifications. If we had skipped that step, our script would have encountered a runtime error.
2. **Did not** call `case.submit`. This is crucial, as `meristem` handles this part.


`meristem` runs the script in "strict mode" (`set -euo pipefail`). If any command in it fails, such as forgetting to `cd` into the case directory first, or a typo in an `xmlchange` variable name, the script stops and that member's stage is marked as failed, rather than the case being submitted with only some of your changes applied. We'll see how to find and retry failed members later in this notebook.

<div class="alert alert-info">
<strong>Exercise: Add output to every member</strong><br><br>

Suppose we also want every ensemble member to write <code>FATES_NPP_PF</code> and <code>FATES_GPP_PF</code> to its primary history file (h0).

How would you change the script above so that each member's <code>user_nl_clm</code> requests these variables?

<em>Hint:</em> the script runs inside each member's case directory (after <code>cd ${case_root}</code>), so anything you could do by hand in a case directory, you can do here.
</div>

<div class="alert alert-success">
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>

Add a line to the end of the script that appends to the clone's <code>user_nl_clm</code>:

```bash
echo "hist_fincl1 = 'FATES_NPP_PF','FATES_GPP_PF'" >> user_nl_clm
```

Because the script runs on each clone, this changes only the members' <code>user_nl_clm</code> files; <code>BONA.FATESsp</code> itself is left untouched. <code>meristem</code> then adds the member's <code>fates_paramfile</code> afterwards.

Note the <code>+=</code>: each clone starts with a copy of the base case's <code>user_nl_clm</code>. If <code>hist_fincl1</code> were already set there, a second <code>hist_fincl1 = ...</code> would silently replace it, while <code>+=</code> adds our variables to the existing list.

</details>
</div>

</details>
</div>

## Step 3: Write an ensemble YAML file

`meristem.ensemble` reads what it needs to know about an ensemble from a YAML configuration file. This file tells `meristem`:

- **what to call the ensemble** and **where to keep track of it** (`ensemble_id`, `root`)
- **which case to clone** for each member (`base_case`)
- **where to find the parameter files**, and which `user_nl_clm` variable to point at them (`paramgen`, `paramfile_key`)
- **what each member should run** (`member_stages`)

Once the ensemble is created, `meristem` saves a copy of this configuration in the ensemble's `root` directory, so the YAML file is only needed once.


Let's look at each part:

| Key | What it does | Our value |
|---|---|---|
| `ensemble_id` | A short name for the ensemble. It is used in job names and in every member's case name. | `oat` |
| `root` | Where `meristem` keeps the ensemble's state files, job scripts, and logs. Member cases are created in `<root>/cases`. | `/glade/u/home/$USER/cases/oat_ensemble` |
| `project` | The project code to charge the jobs to. | your project code |
| `base_case` | The built case every member is cloned from. | `BONA.FATESsp` |
| `paramfile_key` | The `user_nl_clm` variable `meristem` sets to each member's parameter file: `paramfile` for CLM parameter files, `fates_paramfile` for FATES parameter files. | `fates_paramfile` |
| `paramgen` | Where the `paramgen` output lives (`dir`), the prefix of its files (`prefix`), and which members to run (`members`). | `fates_oat`, all members |
| `member_stages` | The stages each member runs, in order. Each one has a `name`, a PBS `queue` for its setup job, and an optional `script` (the one we just wrote). | one stage, `run` |

Each member's case will be named `<base case>_<ensemble_id>_<member>_<stage>`, so the first member's case will be `BONA.FATESsp_oat_fates_oat_0000_run`.

The cell below writes this file for us. Be sure to replace `PXXXXXXX` with your project code.

In [ ]:
config = f"""\
ensemble_id: oat
root: /glade/u/home/{user}/cases/oat_ensemble
project: PXXXXXXX

base_case: /glade/u/home/{user}/cases/BONA.FATESsp
paramfile_key: fates_paramfile

paramgen:
  dir: /glade/u/home/{user}/paramfiles_OAT
  prefix: fates_oat
  members: all

member_stages:
  - name: run
    script: /glade/u/home/{user}/code/setup_oat.sh
    queue: develop
"""
config_path = Path(f"/glade/u/home/{user}/ensemble.yaml")
config_path.write_text(config)
print(config_path.read_text(encoding='utf-8'))

## Step 4: Create the ensemble

Create the ensemble with:

```bash
cd /glade/u/home/$USER

conda activate meristem
meristem ensemble create --config ensemble.yaml
```

Run this in a Derecho terminal, like the rest of the command-line steps

This step submits nothing yet, and only creates the initial files and directories needed to set up the ensemble.